In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [0]:
spark = SparkSession.builder.getOrCreate()

In [0]:
customers_data = [
    (1, "John Doe", "john@example.com", "Hyderabad"),
    (2, "Alice ", "alice@example.com", "Chennai"),
    (3, None, "bob@example.com", "Bangalore"),       
    (4, "David", None, "Mumbai"),                   
    (5, "Eva", "eva@example.com", "Hyderabad"),
    (6, "Frank", "frank@example.com", "Delhi"),
]
customers = spark.createDataFrame(customers_data, ["customer_id", "name", "email", "city"])


In [0]:
orders_data = [
    (101, 1, "2024-01-01", 1000),
    (102, 2, "2024-01-02", 2000),
    (103, 3, "2024-01-03", -500),    
    (104, 99, "2024-01-04", 1500),    
    (105, 1, "2024-01-05", None),    
    (106, 5, "2024-01-06", 3000),
    (107, 5, "2024-01-07", 3000),    
]
orders = spark.createDataFrame(orders_data, ["order_id", "customer_id", "order_date", "amount"])


In [0]:
orders=orders.withColumn("order_date",to_date(col("order_date")))
orders.show()

+--------+-----------+----------+------+
|order_id|customer_id|order_date|amount|
+--------+-----------+----------+------+
|     101|          1|2024-01-01|  1000|
|     102|          2|2024-01-02|  2000|
|     103|          3|2024-01-03|  -500|
|     104|         99|2024-01-04|  1500|
|     105|          1|2024-01-05|  NULL|
|     106|          5|2024-01-06|  3000|
|     107|          5|2024-01-07|  3000|
+--------+-----------+----------+------+



In [0]:
orders.display()
customers.display()

order_id,customer_id,order_date,amount
101,1,2024-01-01,1000
102,2,2024-01-02,2000
103,3,2024-01-03,-500
104,99,2024-01-04,1500
105,1,2024-01-05,null
106,5,2024-01-06,3000
107,5,2024-01-07,3000


customer_id,name,email,city
1,John Doe,john@example.com,Hyderabad
2,Alice,alice@example.com,Chennai
3,null,bob@example.com,Bangalore
4,David,null,Mumbai
5,Eva,eva@example.com,Hyderabad
6,Frank,frank@example.com,Delhi


In [0]:
orders=orders.fillna({'amount':0})
orders.display()

order_id,customer_id,order_date,amount
101,1,2024-01-01,1000
102,2,2024-01-02,2000
103,3,2024-01-03,-500
104,99,2024-01-04,1500
105,1,2024-01-05,0
106,5,2024-01-06,3000
107,5,2024-01-07,3000


In [0]:
orders=orders.withColumn('amount',
                    when(col('amount')<0,0).otherwise(col('amount')))
orders.display()

order_id,customer_id,order_date,amount
101,1,2024-01-01,1000
102,2,2024-01-02,2000
103,3,2024-01-03,0
104,99,2024-01-04,1500
105,1,2024-01-05,0
106,5,2024-01-06,3000
107,5,2024-01-07,3000


In [0]:
customers=customers.fillna({'name':'unknown'}).fillna({'email':'unknown'})
customers.display()

customer_id,name,email,city
1,John Doe,john@example.com,Hyderabad
2,Alice,alice@example.com,Chennai
3,unknown,bob@example.com,Bangalore
4,David,unknown,Mumbai
5,Eva,eva@example.com,Hyderabad
6,Frank,frank@example.com,Delhi


In [0]:
from pyspark.sql.functions import col, regexp_replace, trim

customers = customers.withColumn(
    "name",
    regexp_replace(
        trim(col("name")),
        " ",
        ""
    )
)
display(customers)

customer_id,name,email,city
1,JohnDoe,john@example.com,Hyderabad
2,Alice,alice@example.com,Chennai
3,unknown,bob@example.com,Bangalore
4,David,unknown,Mumbai
5,Eva,eva@example.com,Hyderabad
6,Frank,frank@example.com,Delhi


In [0]:
from pyspark.sql.functions import col, regexp_replace, trim

customers = customers.withColumn(
    "name",
    regexp_replace(
        trim(col("name")),
        " ",
        ""
    )
)
display(customers)

customer_id,name,email,city
1,JohnDoe,john@example.com,Hyderabad
2,Alice,alice@example.com,Chennai
3,unknown,bob@example.com,Bangalore
4,David,unknown,Mumbai
5,Eva,eva@example.com,Hyderabad
6,Frank,frank@example.com,Delhi


In [0]:
combine=customers.join(orders,customers.customer_id==orders.customer_id,how="left_anti")
combine.display()

customer_id,name,email,city
4,David,unknown,Mumbai
6,Frank,frank@example.com,Delhi


In [0]:
combine=customers.join(orders,customers.customer_id==orders.customer_id,how="inner")
combine.display()

customer_id,name,email,city,order_id,customer_id,order_date,amount
1,JohnDoe,john@example.com,Hyderabad,105,1,2024-01-05,0
1,JohnDoe,john@example.com,Hyderabad,101,1,2024-01-01,1000
2,Alice,alice@example.com,Chennai,102,2,2024-01-02,2000
3,unknown,bob@example.com,Bangalore,103,3,2024-01-03,0
5,Eva,eva@example.com,Hyderabad,107,5,2024-01-07,3000
5,Eva,eva@example.com,Hyderabad,106,5,2024-01-06,3000


In [0]:
combine=customers.join(orders,customers.customer_id==orders.customer_id,how="left")
combine=combine.drop(orders.customer_id)
combine.display()

customer_id,name,email,city,order_id,order_date,amount
1,JohnDoe,john@example.com,Hyderabad,105,2024-01-05,0
1,JohnDoe,john@example.com,Hyderabad,101,2024-01-01,1000
2,Alice,alice@example.com,Chennai,102,2024-01-02,2000
3,unknown,bob@example.com,Bangalore,103,2024-01-03,0
4,David,unknown,Mumbai,null,null,null
5,Eva,eva@example.com,Hyderabad,107,2024-01-07,3000
5,Eva,eva@example.com,Hyderabad,106,2024-01-06,3000
6,Frank,frank@example.com,Delhi,null,null,null


In [0]:
combine=combine.fillna({"amount":0})
combine = combine.withColumn(
    "order_date",
    when(col("order_date").isNull(), lit("2024-01-01"))
    .otherwise(col("order_date"))
)
combine.display()

customer_id,name,email,city,order_id,order_date,amount
1,JohnDoe,john@example.com,Hyderabad,105,2024-01-05,0
1,JohnDoe,john@example.com,Hyderabad,101,2024-01-01,1000
2,Alice,alice@example.com,Chennai,102,2024-01-02,2000
3,unknown,bob@example.com,Bangalore,103,2024-01-03,0
4,David,unknown,Mumbai,null,2024-01-01,0
5,Eva,eva@example.com,Hyderabad,107,2024-01-07,3000
5,Eva,eva@example.com,Hyderabad,106,2024-01-06,3000
6,Frank,frank@example.com,Delhi,null,2024-01-01,0


In [0]:
combine.columns

['customer_id', 'name', 'email', 'city', 'order_id', 'order_date', 'amount']

In [0]:
from pyspark.sql.functions import sum

combine = combine.groupBy("customer_id", "name") \
                 .agg(sum("amount").alias("total_spend"))

combine.display()

customer_id,name,total_spend
1,JohnDoe,1000
2,Alice,2000
3,unknown,0
4,David,0
5,Eva,6000
6,Frank,0


In [0]:

from pyspark.sql.window import Window

df_ranked = combine.withColumn(
    "rank",
    rank().over(Window.orderBy(col("total_spend").desc()))
)
df_ranked.display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,name,total_spend,rank
5,Eva,6000,1
2,Alice,2000,2
1,JohnDoe,1000,3
3,unknown,0,4
4,David,0,4
6,Frank,0,4
